# Week 4 — Model Evaluation and Validation
## Employee Attrition Prediction

Evaluates Logistic Regression, Decision Tree and Random Forest using classification metrics, confusion matrices, ROC-AUC, stratified 5-fold cross-validation, generalization checks and error analysis. Hyperparameter tuning is reserved for Week 5.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_curve
RANDOM_STATE=42
RAW_PATH="https://raw.githubusercontent.com/rachitgupt04/employee-attrition-prediction/main/WA_Fn-UseC_-HR-Employee-Attrition.csv"

In [ ]:
try:
    df=pd.read_csv(RAW_PATH)
    print("Dataset loaded successfully:", df.shape)
except Exception as e:
    raise RuntimeError("Dataset could not be loaded. Check the GitHub Raw URL.") from e
if "Attrition" not in df.columns:
    raise ValueError("Required target column Attrition is missing.")
display(df.head())

In [ ]:
data=df.copy()
drop_cols=[c for c in ["EmployeeNumber","EmployeeCount","Over18","StandardHours"] if c in data.columns]
data=data.drop(columns=drop_cols)
data["Attrition"]=data["Attrition"].map({"No":0,"Yes":1})
if data["Attrition"].isna().any(): raise ValueError("Unexpected Attrition values found.")
X=data.drop(columns="Attrition"); y=data["Attrition"].astype(int)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,random_state=42,stratify=y)
num=X_train.select_dtypes(include=np.number).columns.tolist()
cat=X_train.select_dtypes(exclude=np.number).columns.tolist()
preprocessor=ColumnTransformer([("num",Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler())]),num),("cat",Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("onehot",OneHotEncoder(handle_unknown="ignore"))]),cat)])
print("Train:",X_train.shape," Test:",X_test.shape)

In [ ]:
models={"Logistic Regression":LogisticRegression(max_iter=1000,random_state=42),"Decision Tree":DecisionTreeClassifier(random_state=42),"Random Forest":RandomForestClassifier(n_estimators=100,random_state=42,n_jobs=-1)}
pipelines={}; predictions={}; probabilities={}
for name,model in models.items():
    pipelines[name]=Pipeline([("preprocessor",preprocessor),("classifier",model)])
    try:
        pipelines[name].fit(X_train,y_train)
        predictions[name]=pipelines[name].predict(X_test)
        probabilities[name]=pipelines[name].predict_proba(X_test)[:,1]
        print(name,"trained successfully")
    except Exception as e: raise RuntimeError(f"{name} failed") from e

In [ ]:
rows=[]
for name in models:
    p=predictions[name]; prob=probabilities[name]
    rows.append({"Model":name,"Accuracy":accuracy_score(y_test,p),"Precision":precision_score(y_test,p,zero_division=0),"Recall":recall_score(y_test,p,zero_division=0),"F1 Score":f1_score(y_test,p,zero_division=0),"ROC-AUC":roc_auc_score(y_test,prob)})
results=pd.DataFrame(rows).sort_values("F1 Score",ascending=False)
display(results.round(4))

In [ ]:
for name in models:
    print("\n"+"="*60+"\n"+name)
    print(classification_report(y_test,predictions[name],target_names=["No Attrition","Attrition"],zero_division=0))

In [ ]:
for name in models:
    ConfusionMatrixDisplay.from_predictions(y_test,predictions[name],display_labels=["No Attrition","Attrition"],cmap="Blues")
    plt.title("Confusion Matrix — "+name); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(7,5))
for name in models:
    fpr,tpr,_=roc_curve(y_test,probabilities[name]); auc=roc_auc_score(y_test,probabilities[name])
    plt.plot(fpr,tpr,label=f"{name} (AUC={auc:.3f})")
plt.plot([0,1],[0,1],"--",label="Random baseline"); plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate"); plt.title("ROC Curve Comparison"); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
cv_rows=[]
for name in models:
    scores=cross_val_score(pipelines[name],X,y,cv=cv,scoring="f1")
    cv_rows.append({"Model":name,"Mean CV F1":scores.mean(),"Std CV F1":scores.std()})
cv_results=pd.DataFrame(cv_rows).sort_values("Mean CV F1",ascending=False)
display(cv_results.round(4))

In [ ]:
gen=[]
for name in models:
    train_pred=pipelines[name].predict(X_train); test_pred=predictions[name]
    tr=accuracy_score(y_train,train_pred); te=accuracy_score(y_test,test_pred)
    gen.append({"Model":name,"Training Accuracy":tr,"Test Accuracy":te,"Accuracy Gap":tr-te})
generalization=pd.DataFrame(gen); display(generalization.round(4))
generalization.set_index("Model")[["Training Accuracy","Test Accuracy"]].plot(kind="bar",figsize=(8,5)); plt.ylabel("Accuracy"); plt.title("Training vs Test Accuracy"); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

In [ ]:
selected="Random Forest"
errors=X_test.copy(); errors["Actual"]=y_test.values; errors["Predicted"]=predictions[selected]
errors["Error Type"]=np.where((errors.Actual==1)&(errors.Predicted==0),"False Negative",np.where((errors.Actual==0)&(errors.Predicted==1),"False Positive","Correct"))
display(errors["Error Type"].value_counts()); display(errors[errors["Error Type"]=="False Negative"].head(10))

### Interpretation
Use the executed values above for the final report. Do not select a model from accuracy alone. Pay particular attention to Attrition recall and F1-score. A large training/test gap can indicate overfitting, but should be interpreted with cross-validation and other metrics. Week 5 will cover hyperparameter optimization.